In [44]:
from dotenv import load_dotenv
import os
load_dotenv()
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")
USERNAME = os.getenv("USERNAME")
PASSWORD = os.getenv("PASSWORD")
DATABASE = os.getenv("DATABASE")

In [45]:
import mysql.connector
from mysql.connector import Error

connection = mysql.connector.connect(
    host=HOST,
    port=PORT,
    user=USERNAME,
    password=PASSWORD,
    database=DATABASE
)

cursor = connection.cursor()
print("Connected:", connection.is_connected())

Connected: True


In [46]:
cursor.execute("SHOW TABLES;")
tables = [row[0] for row in cursor.fetchall()]
print(f"Found {len(tables)} tables:")
for t in tables:
    print(" -", t)

Found 4 tables:
 - catalogue
 - landcover
 - terrain
 - training_table


In [47]:
schema = {}

for table in tables:
    cursor.execute(f"""
        SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE, COLUMN_KEY, COLUMN_DEFAULT
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = DATABASE()
        AND TABLE_NAME = %s
        ORDER BY ORDINAL_POSITION;
    """, (table,))
    
    columns = cursor.fetchall()
    schema[table] = columns

    print(f"\nTable: {table}")
    for col_name, data_type, nullable, key, default in columns:
        key_marker = f" [{key}]" if key else ""
        print(f"  - {col_name} ({data_type}){key_marker}")


Table: catalogue
  - collection_id (varchar) [PRI]
  - title (varchar)
  - constellation (varchar)
  - instruments (varchar)
  - processing_level (varchar)
  - gsd_min_m (decimal)
  - gsd_max_m (decimal)
  - n_bands (smallint)
  - band_names (text)
  - temporal_start (datetime)
  - temporal_end (datetime)
  - bbox (varchar)
  - license (varchar)
  - doi (varchar)
  - providers (varchar)
  - keywords (varchar)

Table: landcover
  - id (int) [PRI]
  - longitude (decimal) [MUL]
  - latitude (decimal)
  - class_code (tinyint)
  - class_name (varchar)

Table: terrain
  - id (int) [PRI]
  - longitude (decimal) [MUL]
  - latitude (decimal)
  - elevation_m (decimal)

Table: training_table
  - id (int) [PRI]
  - longitude (decimal) [MUL]
  - latitude (decimal)
  - elevation_m (decimal)
  - slope_deg (decimal)
  - land_cover_code (tinyint)
  - land_cover_class (varchar)
  - is_built_up (tinyint)


In [48]:
import pandas as pd

In [49]:
rows = []
for table, columns in schema.items():
    for col_name, data_type, nullable, key, default in columns:
        rows.append({
            "table": table,
            "column": col_name,
            "type": data_type,
            "nullable": nullable,
            "key": key,
            "default": default
        })

df = pd.DataFrame(rows)
df

,table,column,type,nullable,key,default
0,catalogue,collection_id,varchar,NO,PRI,NaN
1,catalogue,title,varchar,NO,,NaN
2,catalogue,constellation,varchar,YES,,NULL
3,catalogue,instruments,varchar,YES,,NULL
4,catalogue,processing_level,varchar,YES,,NULL
5,catalogue,gsd_min_m,decimal,YES,,NULL
6,catalogue,gsd_max_m,decimal,YES,,NULL
7,catalogue,n_bands,smallint,NO,,NaN
8,catalogue,band_names,text,YES,,NULL
9,catalogue,temporal_start,datetime,NO,,NaN


In [52]:
query = "SELECT * FROM catalogue"
df = pd.read_sql(query, connection)
df

/tmp/ipykernel_128606/3119065351.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


,collection_id,title,constellation,instruments,processing_level,gsd_min_m,gsd_max_m,n_bands,band_names,temporal_start,temporal_end,bbox,license,doi,providers,keywords
0,ccm-optical,Copernicus Contributing Missions Optical,alos;beijing;dmc;geosat;worldview;formosat;geo...,aeiss;avnir-2;awf;bgi;dem;dove;gis;hirais;hr;h...,L1;L2,0.5,66.0,0,,1998-01-19 14:44:39,NaT,"-83.711914,-35.5704748260475,65.787712,80.984856",other,,ESA;CloudFerro,Copernicus;ESA;EU;Imagery;Optical;Reflectance;...
1,ccm-sar,Copernicus Contributing Missions SAR,cosmo-skymed;iceye;paz;radarsat;terrasar-x,sar;x-sar;xsar1,,0.0,0.0,0,,2011-05-31 06:26:33,NaT,"-179.95546,-81.05,179.96101,143.99498",other,,ESA;CloudFerro,Copernicus;ESA;EU;Imagery;SAR;Radar;Satellite;...
2,clms_ba_global_300m_daily_v3_cog,CLMS Burnt Area (BA) Global 300m daily V3 (COG),sentinel-3,olci;slstr,L3,300.0,300.0,1,dob_nrt,2023-07-01 00:00:00,2025-11-01,"-179.9999999,-59.9985119,179.9985119,80.0014881",other,10.2909/9c0519f9-d2c2-4469-a9e1-2222d37c33d6,CloudFerro;European Environment Agency;Europea...,Copernicus;CLMS;BA;Burnt Area;fire disturbance...
3,clms_ba_global_300m_daily_v3_nc,CLMS Burnt Area (BA) Global 300m daily V3 (Net...,sentinel-3,olci;slstr,L3,300.0,300.0,1,day_of_burn,2023-07-01 00:00:00,2025-11-01,"-179.9999999,-59.9985119,179.9985119,80.0014881",other,10.2909/9c0519f9-d2c2-4469-a9e1-2222d37c33d6,CloudFerro;European Environment Agency;Europea...,Copernicus;CLMS;BA;Burnt Area;fire disturbance...
4,clms_ba_global_300m_daily_v4_cog,CLMS Burnt Area (BA) Global 300m daily V4 (COG),sentinel-3,olci;slstr,L3,300.0,300.0,4,cp_nrt;bf_nrt;dob_nrt;lfp_nrt,2024-12-01 00:00:00,NaT,"-179.9999999,-59.9985119,179.9985119,80.0014881",other,10.2909/bfd77180-7d7c-4c1c-b193-1489f735d5f1,CloudFerro;European Environment Agency;Europea...,Copernicus;CLMS;BA;Burnt Area;fire disturbance...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,sentinel-6-p4-1b-ntc,Sentinel-6 P4 Level 1B Non Time Critical (NTC),sentinel-6,p4,L1,20000.0,20000.0,2,;,2021-12-16 17:38:58,NaT,"-180,-90,180,90",other,,EUMETSAT;European Commission;CloudFerro,Copernicus;Sentinel;EU;ESA;EUMETSAT;Satellite;...
414,sentinel-6-p4-1b-stc,Sentinel-6 P4 Level 1B Short Time Critical (STC),sentinel-6,p4,L1,20000.0,20000.0,2,;,2021-10-09 10:13:44,NaT,"-180,-90,180,90",other,,EUMETSAT;European Commission;CloudFerro,Copernicus;Sentinel;EU;ESA;EUMETSAT;Satellite;...
415,sentinel-6-p4-2-nrt,Sentinel-6 P4 Level 2 Near Real-Time (NRT),sentinel-6,p4,L2,20000.0,20000.0,2,;,2021-10-09 05:15:37,NaT,"-180,-90,180,90",other,,EUMETSAT;European Commission;CloudFerro,Copernicus;Sentinel;EU;ESA;EUMETSAT;Satellite;...
416,sentinel-6-p4-2-ntc,Sentinel-6 P4 Level 2 Non Time Critical (NTC),sentinel-6,p4,L2,20000.0,20000.0,2,;,2021-10-30 00:19:22,NaT,"-180,-90,180,90",other,,EUMETSAT;European Commission;CloudFerro,Copernicus;Sentinel;EU;ESA;EUMETSAT;Satellite;...


In [51]:
for table in schema.items():
    number += 1
    query_table = f"SELECT * FROM {table}"
    df = pd.read_sql(query, connection)
    

NameError: name 'number' is not defined

In [ ]:
df

,id,longitude,latitude,elevation_m
0,1,12.500208,55.699861,36.19
1,2,12.500625,55.699861,36.39
2,3,12.501042,55.699861,34.77
3,4,12.501458,55.699861,36.19
4,5,12.501875,55.699861,38.23
...,...,...,...,...
34555,34556,12.578125,55.650139,3.69
34556,34557,12.578542,55.650139,3.46
34557,34558,12.578958,55.650139,4.00
34558,34559,12.579375,55.650139,6.22


In [ ]:
df2

NameError: name 'df2' is not defined